In [1]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [ ]:
INDEX_PATH = cfg["paths"]["index"]
print("Source index:", INDEX_PATH)

In [ ]:
# Load the main index.
LAYER = "photos"
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

### Examine the available years and projects
The second cell below displays unique combos between flight codes and 

In [4]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


In [5]:
# Examine unique year + flight code combinations for the county
county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

flights = (
    county_gdf[["year", "flight_code"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["year", "flight_code"])
)

for year, flight_code in flights.itertuples(index=False, name=None):
    print(f"{year}: {flight_code}")

1937: Carver
1937: Hennepin
1937: Scott
1938: bjm
1940: Carver
1940: McLeod
1940: Scott
1947: dv
1951: Carver
1951: Scott
1951: wm
1953: Hennepin
1955: wl
1956: Hennepin
1957: Carver
1957: Hennepin
1957: Scott
1960: Hennepin
1962: Hennepin
1963: Carver
1964: Hennepin
1964: Scott
1967: Hennepin
1969: Hennepin
1971: Hennepin


In [6]:
county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

county_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
13511,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/carver/y1937/wm-26-236...,Carver,91674,None,None,None,None,None,44.729640,POINT (-10465155.126 5579058.785)
13512,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/carver/y1937/wm-26-236...,Carver,91675,None,None,None,None,None,44.741189,POINT (-10465099.911 5580868.601)
13513,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/carver/y1937/wm-26-237...,Carver,91676,None,None,None,None,None,44.752738,POINT (-10465044.585 5582678.778)
13514,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/carver/y1937/wm-26-237...,Carver,91677,None,None,None,None,None,44.764287,POINT (-10464989.371 5584489.317)
13515,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/carver/y1937/wm-26-237...,Carver,91678,None,None,None,None,None,44.775835,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14510,carver_1971_clk-1-134,None,Carver,1971,clk-1-134,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/hennepin_county/y1971/...,Hennepin,95187,None,None,None,None,None,44.880391,POINT (-10416624.28 5602711.11)
14511,carver_1971_clk-1-46,None,Carver,1971,clk-1-46,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/hennepin_county/y1971/...,Hennepin,95188,None,None,None,None,None,44.880654,POINT (-10414425.386 5602752.428)
14512,carver_1971_clk-1-47,None,Carver,1971,clk-1-47,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/hennepin_county/y1971/...,Hennepin,95189,None,None,None,None,None,44.889025,POINT (-10414420.488 5604067.63)
14513,carver_1971_clk-1-44,None,Carver,1971,clk-1-44,NaN,NaN,NaN,NaN,None,...,https://geo.lib.umn.edu/hennepin_county/y1971/...,Hennepin,95268,None,None,None,None,None,44.885348,POINT (-10412050.83 5603489.898)


### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Compare this to the list above to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- **Choose a year or multiple years with the same scale**
- Specify the year(s) and scale for your current batch in config.yml.
- The batch will be named for the county, years, and scale. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


In [ ]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [ ]:
batch_gdf = index_gdf.loc[mask].copy()

In [2]:

TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"
print("Batch tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg
Batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1957_tracker.gpkg
County: Carver
Years: [1957]
Scale: 20000


## Create a tracker table

In [ ]:
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [ ]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

In [ ]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [ ]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

In [ ]:
batch_gdf.columns

In [ ]:
if TRACKER_PATH.exists():
    raise FileExistsError(
        f"Batch tracker already exists: {TRACKER_PATH}"
    )

TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

batch_gdf.to_file(
    TRACKER_PATH,
    layer=LAYER,
    driver="GPKG",
)

print(f"Created batch tracker: {TRACKER_PATH}")

## Download JPEGS and calculate extents

In [ ]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

In [ ]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

In [ ]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

In [ ]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

### Final load & preview

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf